<div dir="rtl">

# 🚀 04 - End-to-End Web RAG Application (GenAI Application)

## نبذة عن المشروع المتكامل:
في هذا الكراس الختامي، سنقوم ببناء **تطبيق ذكاء اصطناعي توليدي كامل (End-to-End GenAI App)** يمر بكافة مراحل معالجة واسترجاع وتوليد الإجابات من الصفر وحتى واجهة الاستعلام التفاعلية:
1. **استيعاب البيانات (Data Ingestion)**: سحب وقراءة وثائق رسمية مباشرة من الإنترنت عبر `WebBaseLoader`.
2. **التقسيم الذكي (Text Splitting)**: تقطيع النصوص مع الحفاظ على ترابط الفقرات عبر `RecursiveCharacterTextSplitter`.
3. **التضمين المتجهي (Embeddings)**: توليد التضمينات محلياً ومجانياً باستخدام `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`).
4. **الفهرسة والاسترجاع (Vector Store & Retrieval)**: إنشاء فهرس `FAISS` عالي السرعة.
5. **سلسلة التوليد الذكي (LCEL RAG Pipeline)**: ربط خط الأنابيب بنموذج `ChatGroq` مع مواءمة السياق وتوليد الإجابات الموثقة.

</div>

<div dir="rtl">

### ⚙️ الخطوة 0: تهيئة البيئة والتحقق من المفاتيح

</div>

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# تحميل متغيرات البيئة
load_dotenv(find_dotenv())

groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    print("✅ تم التحقق من مفتاح GROQ_API_KEY بنجاح!")
else:
    print("⚠️ تحذير: يرجى التأكد من تعريف GROQ_API_KEY في ملف .env")


<div dir="rtl">

### 1️⃣ استيعاب البيانات من الويب (Web Data Ingestion)
نستخدم `WebBaseLoader` لجلب محتوى صفحة توثيق حقيقية من موقع LangSmith الرسمي وتنقيتها.

</div>

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

target_url = "https://docs.smith.langchain.com/tutorials/Administrators/manage_spend"
print(f"🌐 جاري استيراد البيانات من الرابط: {target_url}")

loader = WebBaseLoader(target_url)
raw_docs = loader.load()

print(f"✅ تم استيراد المستند بنجاح!")
print(f"📄 العنوان: {raw_docs[0].metadata.get('title', 'N/A')}")
print(f"📊 إجمالي عدد الحروف المستخرجة: {len(raw_docs[0].page_content)} حرفاً")


<div dir="rtl">

### 2️⃣ تقسيم النصوص إلى قطع دلالية (Chunking)
نستخدم `RecursiveCharacterTextSplitter` لتقطيع النصوص مع تداخل مناسب (`chunk_size=1000`, `chunk_overlap=200`).

</div>

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = text_splitter.split_documents(raw_docs)

print(f"✂️ تم تقسيم المستند إلى: {len(chunks)} قطعة نصية (Chunks)")
print(f"🔍 عينة من القطعة الأولى:\n{chunks[0].page_content[:300]}...")


<div dir="rtl">

### 3️⃣ بناء مستودع FAISS وتضمين النصوص محلياً
توليد متجهات التضمين باستخدام نموذج `all-MiniLM-L6-v2` وتخزينها في فهرس `FAISS` فائق السرعة.

</div>

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# تهيئة نموذج التضمين
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# بناء فهرس المتجهات
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("✅ تم بناء مستودع FAISS وتجهيز الـ Retriever بنجاح!")


<div dir="rtl">

### 4️⃣ اختبار البحث الدلالي الأولي (Similarity Search Test)
التحقق من قدرة المستودع على جلب المقاطع الأكثر صلة بالاستعلام قبل دمجها مع النموذج التوليدي.

</div>

In [ ]:
test_query = "How can administrators monitor and manage spend in LangSmith?"
matched_docs = retriever.invoke(test_query)

print(f"🔎 عدد النتائج المسترجعة: {len(matched_docs)}")
for i, doc in enumerate(matched_docs, 1):
    print(f"\n--- نتيجة {i} ---")
    print(doc.page_content[:250] + "...")


<div dir="rtl">

### 5️⃣ بناء خط أنابيب الـ RAG المتكامل عبر LCEL
نقوم بإنشاء قالب توجيه احترافي يربط الـ Retriever ونموذج `ChatGroq` ومحلل المخرجات `StrOutputParser`.

</div>

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# صياغة القالب الموجه
rag_prompt = ChatPromptTemplate.from_template(
    """You are a world-class AI assistant specialized in LangChain and LangSmith documentation.
Answer the user's question accurately and concisely based strictly on the provided Context below.
If the answer cannot be found in the context, reply: 'I cannot find this information in the provided documentation.'

Context:
{context}

Question: {question}

Grounded Answer:"""
)

# تهيئة نموذج Groq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3
)

# خط أنابيب LCEL المتكامل
rag_app = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("🚀 تم بناء وتجميع تطبيق RAG GenAI بنجاح!")


<div dir="rtl">

### 6️⃣ اختبار التطبيق بأسئلة تفاعلية (Interactive Evaluation)
نقوم باختبار استفسارات حقيقية حول إدارة التكاليف وتتبع الأداء وفحص دقة الإجابة وسرعة التوليد.

</div>

In [ ]:
questions = [
    "What features does LangSmith Observability provide for LLM applications?",
    "How does LangSmith handle tracing and what are its key benefits?"
]

for q in questions:
    print(f"\n🔹 السؤال: {q}")
    response = rag_app.invoke(q)
    print("💬 الإجابة:")
    print(response)
    print("-" * 70)


<div dir="rtl">

## 💡 خلاصة المسار التعليمي (Curriculum Summary)
تهانينا! 🎉 لقد أتممت بنجاح بناء تطبيق ذكاء اصطناعي توليدي كامل (End-to-End GenAI App) يجمع:
1. **سحب بيانات حية** من الويب (`WebBaseLoader`).
2. **التقسيم الذكي** للنصوص (`RecursiveCharacterTextSplitter`).
3. **التضمينات المحلية** السريعة (`HuggingFaceEmbeddings`).
4. **الفهرسة في قاعدة متجهات** (`FAISS`).
5. **معمارية السلاسل الحديثة** بلغة التعبير (`LCEL`) ونماذج `Groq` السريعة لتقديم إجابات ذكية وموثقة.

</div>